In [1]:
import os
os.chdir(r"C:\Users\wangy")
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("diffusion(S3)/final_results.xlsx")
raw = pd.read_excel("diffusion(S3)/diffusion_genderTopicvector.xlsx")

In [3]:
# Step 0 — Prepare raw data for later cascade size
cascade_size = (
    raw.groupby(["agent_name", "article_title"])
        .size()
        .rename("cascade_size")
        .reset_index()
)
# Add cascade size to summary df
df = df.merge(cascade_size, on=["agent_name","article_title"], how="left")

In [4]:
# Step 1 — First-degree width (max, avg)
g = df.groupby("agent_name")

agent_agg = g.agg(
    first_layer_width_max=("first_layer_width", "max"),
    first_layer_width_avg=("first_layer_width", "mean"),
)

# Step 2 — Second-degree width (max, avg)
agent_agg["second_layer_width_max"] = g["second_layer_width"].max()
agent_agg["second_layer_width_avg"] = g["second_layer_width"].mean()

# Step 3 — Cascade size (mean, variance)
agent_agg["cascade_size_mean"] = g["cascade_size"].mean()
agent_agg["cascade_size_var"] = g["cascade_size"].var()

# Step 4 — Depth (mean, variance)
agent_agg["depth_mean"] = g["depth"].mean()
agent_agg["depth_var"] = g["depth"].var()

In [5]:
# Step 5 — 1st-degree repeat exposure (cascade-level then agent avg)
READER_COL = "reader_wechat_nn"

# readers who are in layer 1 at least once (defines the node set)
layer1_nodes = raw[raw["correct_layer"] == 1][["agent_name", READER_COL]].drop_duplicates()

# article membership for readers across ANY layer
raw_any1 = raw[["agent_name", "article_title", READER_COL]].drop_duplicates()

total_articles_any1 = (
    raw_any1.groupby("agent_name")["article_title"].nunique()
           .rename("total_articles_any1")
           .reset_index()
)

articles_seen_any1 = (
    raw_any1.groupby(["agent_name", READER_COL])["article_title"].nunique()
           .rename("articles_seen_any1")
           .reset_index()
           .merge(total_articles_any1, on="agent_name", how="left")
)

# keep only readers who are layer-1 nodes
tmp1 = (layer1_nodes
       .merge(articles_seen_any1, on=["agent_name", READER_COL], how="left"))

tmp1["is_repeat_1st"] = tmp1["articles_seen_any1"] >= np.ceil(0.3 * tmp1["total_articles_any1"])

repeat_1st_agent = (
    tmp1.groupby("agent_name")["is_repeat_1st"]
       .mean()
       .rename("repeat_exposure_1st_nodes_pct")
)
agent_agg = agent_agg.join(repeat_1st_agent, on="agent_name")

# Step 6 — 2nd-degree repeat exposure (cascade-level then agent avg)
# readers who are in layer 2 at least once (defines the node set)
layer2_nodes = raw[raw["correct_layer"] == 2][["agent_name", READER_COL]].drop_duplicates()

# article membership for readers across ANY layer
raw_any2 = raw[["agent_name", "article_title", READER_COL]].drop_duplicates()

total_articles_any2 = (
    raw_any2.groupby("agent_name")["article_title"].nunique()
           .rename("total_articles_any2")
           .reset_index()
)

articles_seen_any2 = (
    raw_any2.groupby(["agent_name", READER_COL])["article_title"].nunique()
           .rename("articles_seen_any2")
           .reset_index()
           .merge(total_articles_any2, on="agent_name", how="left")
)

# keep only readers who are layer-1 nodes
tmp2 = (layer2_nodes
       .merge(articles_seen_any2, on=["agent_name", READER_COL], how="left"))

tmp2["is_repeat_2nd"] = tmp2["articles_seen_any2"] >= np.ceil(0.3 * tmp2["total_articles_any2"])

repeat_2nd_agent = (
    tmp2.groupby("agent_name")["is_repeat_2nd"]
       .mean()
       .rename("repeat_exposure_2nd_nodes_pct")
)
agent_agg = agent_agg.join(repeat_2nd_agent, on="agent_name")


In [6]:
# Step 7 — Reshare % (mean, variance)
agent_agg["reshare_mean"] = g["reshare_pct"].mean()
agent_agg["reshare_var"] = g["reshare_pct"].var()

# Step 8–10 — structural virality, centrality, Wiener index
agent_agg["structural_virality_mean"] = g["structural_virality"].mean()
agent_agg["centrality_mean"] = g["centrality"].mean()
agent_agg["agent_deg_centrality_mean"] = g["agent_deg_centrality"].mean()
agent_agg["avg_out_degree_centrality_mean"] = g["avg_out_degree_centrality"].mean()
agent_agg["wiener_index_mean"] = g["wiener_index"].mean()

# Step 11 — Gender %
agent_agg["gender_pct_all_1_mean"] = g["gender_pct_all_1"].mean()
agent_agg["gender_pct_all_0_mean"] = g["gender_pct_all_0"].mean()

# Step 12 — Gender homophily
agent_agg["gender_assortativity_mean"] = g["gender_assortativity"].mean()

# Step 13 — Profession–content match (mean)
agent_agg["matchscore_mean"] = g["MatchScore"].mean()
agent_agg["profession_content_match_mean"] = g["ProfessionContentMatch"].mean()

# Step 14 — Reader time duration (mean, variance)
agent_agg["duration_mean_of_means"] = g["duration_mean_s"].mean()
agent_agg["duration_var_of_means"] = g["duration_mean_s"].var()

agent_agg["duration_mean_of_vars"] = g["duration_var_s2"].mean()
agent_agg["duration_var_of_vars"] = g["duration_var_s2"].var()


In [7]:
agent_agg.head()

,first_layer_width_max,first_layer_width_avg,second_layer_width_max,second_layer_width_avg,cascade_size_mean,cascade_size_var,depth_mean,depth_var,repeat_exposure_1st_nodes_pct,repeat_exposure_2nd_nodes_pct,...,wiener_index_mean,gender_pct_all_1_mean,gender_pct_all_0_mean,gender_assortativity_mean,matchscore_mean,profession_content_match_mean,duration_mean_of_means,duration_var_of_means,duration_mean_of_vars,duration_var_of_vars
agent_name,,,,,,,,,,,,,,,,,,,,,
+章珊,21,4.366667,5,0.200000,6.033333,21.481609,0.900000,0.231034,0.015873,0.000000,...,37.700000,0.342428,0.657572,-0.100595,0.331000,0.133333,713.495421,763122.989552,1.801058e+06,9.619173e+12
丁丽美,13,7.666667,0,0.000000,8.166667,26.166667,1.000000,0.000000,0.194444,NaN,...,78.666667,0.348291,0.651709,-0.067000,0.468750,0.333333,562.019231,341582.008136,3.737860e+06,5.257878e+13
丁妮,37,11.908046,29,2.252874,18.804598,969.996258,1.574713,1.386795,0.000000,0.000000,...,2462.011494,0.455023,0.544977,-0.091121,0.302759,0.068966,1022.519194,770054.557851,4.411732e+06,1.237126e+14
丁金雄,7,4.000000,8,3.600000,9.000000,16.000000,2.400000,0.800000,0.500000,0.117647,...,115.600000,0.515000,0.485000,-0.038869,0.577400,0.600000,329.250000,147574.687500,6.996375e+05,1.012430e+12
万延和,30,10.333333,98,20.333333,108.833333,52032.566667,2.833333,12.566667,0.225806,0.010526,...,107121.500000,0.569335,0.430665,-0.070023,0.428333,0.333333,407.558901,169132.088878,3.847811e+05,1.756941e+11


In [8]:
# Replace all NaN with 0
agent_agg = agent_agg.fillna(0)

In [9]:
agent_agg = agent_agg.reset_index()
agent_agg.to_excel("diffusion(S3)/agent_level_results.xlsx", index=False)